# 01. Data Profiling & Data Audit

**Project:** Fashion Campus E-commerce Analytics

**Purpose:**  
Understand the structure, grain, relationships, and data quality of the
raw datasets before data cleaning and modeling.

### Profiling Objectives

This notebook aims to:

1. Understand the structure and size of each dataset.
2. Identify the grain and candidate primary keys.
3. Validate relationships between datasets.
4. Detect data quality issues such as missing values, duplicates,
   invalid values, and structural problems.
5. Validate important business rules in the transaction data.
6. Define the cleaning requirements for the next stage.

> No permanent transformation is applied to the raw data during this stage.

## 1. Setup & Load Raw Data

The project uses four raw datasets:

- `customer.csv`
- `product.csv`
- `transaction_new.csv`
- `click_stream_new.csv`

The raw files are loaded without modifying the source data.

In [1]:
from pathlib import Path 
import pandas as pd 

project_root=Path.cwd().parent
raw_dir=project_root/"data"/"raw"
print(f"Project: {project_root}")
print(f"Raw Data: {raw_dir}")
print("Files:")
for file in raw_dir.iterdir():
    print(f"- {file.name}")


Project: c:\Users\LENOVO\fashion-campus-ecommerce-analytics
Raw Data: c:\Users\LENOVO\fashion-campus-ecommerce-analytics\data\raw
Files:
- click_stream_new.csv
- customer.csv
- product.csv
- transaction_new.csv


In [4]:
customer = pd.read_csv(raw_dir / "customer.csv")
print("customer:", customer.shape)

customer: (100000, 15)


In [5]:
transaction = pd.read_csv(raw_dir / "transaction_new.csv")
print("transaction:", transaction.shape)

transaction: (1254585, 16)


In [6]:
click_stream = pd.read_csv(raw_dir / "click_stream_new.csv")
print("click_stream:", click_stream.shape)

click_stream: (12833602, 12)


In [5]:
product = pd.read_csv(raw_dir / "product.csv")
print("product:", product.shape)

ParserError: Error tokenizing data. C error: Expected 10 fields in line 6044, saw 11


## 2. Dataset Overview

The dataset contains four main entities:

| Dataset | Business Entity | Expected Role |
|---|---|---|
| Customer | Customer | Customer master data |
| Product | Product | Product master data |
| Transaction | Orders / Order Items | Purchase transactions |
| Click Stream | Customer Events | Customer journey and engagement |

The initial profiling will examine row counts, columns, uniqueness, and candidate keys.

In [ ]:
datasets = {
    "customer": customer,
    "transaction": transaction,
    "click_stream": click_stream
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customer: (100000, 15)
transaction: (1254585, 16)
click_stream: (12833602, 12)


## 3. Customer Table Profiling


### 3.1 Schema

In [28]:
print("Shape:", customer.shape)
print("\nColumns:")
print(customer.columns.tolist())

print("\nData types:")
print(customer.dtypes)

Shape: (100000, 15)

Columns:
['customer_id', 'first_name', 'last_name', 'username', 'email', 'gender', 'birthdate', 'device_type', 'device_id', 'device_version', 'home_location_lat', 'home_location_long', 'home_location', 'home_country', 'first_join_date']

Data types:
customer_id                    int64
first_name                       str
last_name                        str
username                         str
email                            str
gender                           str
birthdate             datetime64[us]
device_type                      str
device_id                        str
device_version                   str
home_location_lat            float64
home_location_long           float64
home_location                    str
home_country                     str
first_join_date       datetime64[us]
dtype: object


### 3.2 Grain & Primary Key

The expected grain of the customer dataset is:

> 1 row = 1 customer

The candidate primary key is `customer_id`.

In [30]:
print("Rows:", len(customer))
print("Unique customer_id:", customer["customer_id"].nunique())
print("Duplicate customer_id:", customer["customer_id"].duplicated().sum())
print("Unique device_id:", customer["device_id"].nunique())
print("Duplicate device_id:", customer["device_id"].duplicated().sum())

Rows: 100000
Unique customer_id: 100000
Duplicate customer_id: 0
Unique device_id: 100000
Duplicate device_id: 0


In [14]:
customer_device=(customer.groupby("customer_id")["device_id"].nunique())

print("Customers:", customer_device.shape[0])
print(
    "Customers with multiple devices:",
    (customer_device > 1).sum()
)
print(
    "Maximum devices per customer:",
    customer_device.max()
)


Customers: 100000
Customers with multiple devices: 0
Maximum devices per customer: 1


In [15]:
device_customer = (
    customer.groupby("device_id")["customer_id"]
    .nunique()
)

print("Devices:", device_customer.shape[0])
print(
    "Devices with multiple customers:",
    (device_customer > 1).sum()
)
print(
    "Maximum customers per device:",
    device_customer.max()
)

Devices: 100000
Devices with multiple customers: 0
Maximum customers per device: 1


### Finding

The relationship between `customer_id` and `device_id` is 1:1:

- Each customer belongs to exactly one device.
- Each device belongs to exactly one customer.

Although `customer_id` and `device_id` are both unique across all customer records, `customer_id` is the strong candidate for the primary key because `device_id` does not exist in the transaction table. Therefore, the customer table appears to have a customer-level grain.

### 3.3 Data Quality

In [31]:
customer.isna().sum().sort_values(ascending=False)

customer_id           0
first_name            0
last_name             0
username              0
email                 0
gender                0
birthdate             0
device_type           0
device_id             0
device_version        0
home_location_lat     0
home_location_long    0
home_location         0
home_country          0
first_join_date       0
dtype: int64

In [33]:
customer.duplicated().sum()

np.int64(0)

#### Finding
- No missing values.
- No duplicated values.

### 3.4 Key categorical columns


In [36]:
for col in ["gender", "device_type", "home_country"]:
    print(f"\n{col}:")
    print(customer[col].value_counts(dropna=False))


gender:
gender
F    64230
M    35770
Name: count, dtype: int64

device_type:
device_type
Android    76637
iOS        23363
Name: count, dtype: int64

home_country:
home_country
Indonesia    100000
Name: count, dtype: int64


### 3.5 Date Validation

In [38]:
for col in ["birthdate", "first_join_date"]:
    customer[col] = pd.to_datetime(
        customer[col],
        errors="coerce"
    )

    print(f"\n{col}")
    print("Invalid:", customer[col].isna().sum())
    print("Min:", customer[col].min())
    print("Max:", customer[col].max())

print("First Join Date < Birthdate:", (customer["first_join_date"]<customer["birthdate"]).sum())


birthdate
Invalid: 0
Min: 1954-06-30 00:00:00
Max: 2016-07-27 00:00:00

first_join_date
Invalid: 0
Min: 2016-06-30 00:00:00
Max: 2022-07-31 00:00:00
First Join Date < Birthdate: 0


### Customer Profiling Finding

- Candidate primary key: `customer_id`
- Expected grain: 1 row = 1 customer
- Missing-value issues: [to be completed]
- Duplicate issues: [to be completed]
- Date issues: [to be completed]

Any required treatment will be performed during the cleaning stage.

## 4. Product Table Profiling

The product dataset is expected to have the following grain:

> 1 row = 1 product

The candidate primary key is `id`.

During initial loading, the standard CSV parser detects malformed rows.
Therefore, the raw file structure is audited before loading it into a DataFrame.

In [ ]:
product_path = raw_dir / "product.csv"

with open(product_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Physical lines:", len(lines))
print("Header:", lines[0])

Physical lines: 44447
Header: id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName



In [ ]:
malformed_rows = []

expected_fields = 10

for line_number, line in enumerate(lines[1:], start=2):
    field_count = len(line.rstrip("\n").split(","))

    if field_count != expected_fields:
        malformed_rows.append(
            (line_number, field_count, line.strip())
        )

print("Malformed rows:", len(malformed_rows))

Malformed rows: 22


In [ ]:
for row in malformed_rows:
    print(row)

(6044, 11, '44065,Men,Personal Care,Fragrance,Perfume and Body Mist,Blue,Spring,2017,Casual,Boss Men Perfume, After Shave Balm and Shower Gel Set')
(6569, 11, "3413,Men,Apparel,Topwear,Tshirts,White,Summer,2011,Casual,Myntra Men's Yes, its all about me White T-shirt")
(7399, 11, '23101,Men,Accessories,Accessories,Accessory Gift Set,Pink,Fall,2011,Formal,Turtle Men Formal Pink Tie, Cufflink and Pocket Square Combo Set')
(7939, 11, '23108,Men,Accessories,Accessories,Accessory Gift Set,Purple,Fall,2011,Formal,Turtle Men Formal Purple Tie, Cufflink and Pocket Square Combo Set')
(9026, 11, '4854,Unisex,Footwear,Shoes,Casual Shoes,White,Winter,2012,Casual,Newfeel Unisex White, Blue and Orange Casual Shoes')
(10264, 11, '23109,Men,Accessories,Accessories,Accessory Gift Set,Purple,Fall,2011,Formal,Turtle Men Formal Purple Tie, Cufflink and Pocket Square Combo Set')
(10427, 11, '23100,Men,Accessories,Accessories,Accessory Gift Set,Navy Blue,Fall,2011,Formal,Turtle Men Formal Navy Blue Tie, Cuff

### Data Quality Finding

The standard CSV parser cannot load `product.csv` because 22 rows contain unescaped commas within the `productDisplayName` field.

Example:

`Boss Men Perfume, After Shave Balm and Shower Gel Set`

This causes the parser to interpret the product name as multiple fields.

### Impact

- The raw product file cannot be loaded using the default pandas CSV parser.
- The affected rows should not be dropped.
- The issue will be repaired during the data cleaning phase.

> The raw file remains unchanged during profiling.

## 5. Transaction Table Profiling


### 5.1 Schema

In [39]:
print("Shape:", transaction.shape)

print("\nColumns:")
print(transaction.columns.tolist())

print("\nData types:")
print(transaction.dtypes)

Shape: (1254585, 16)

Columns:
['created_at', 'customer_id', 'booking_id', 'session_id', 'payment_method', 'payment_status', 'promo_amount', 'promo_code', 'shipment_fee', 'shipment_date_limit', 'shipment_location_lat', 'shipment_location_long', 'total_amount', 'product_id', 'quantity', 'item_price']

Data types:
created_at                    str
customer_id                 int64
booking_id                    str
session_id                    str
payment_method                str
payment_status                str
promo_amount                int64
promo_code                    str
shipment_fee                int64
shipment_date_limit           str
shipment_location_lat     float64
shipment_location_long    float64
total_amount                int64
product_id                  int64
quantity                    int64
item_price                  int64
dtype: object


### 5.2 Grain & Primary Key

The transaction dataset contains multiple product lines within the same
booking.

The expected grain is:

> 1 row = 1 product line within a booking

Candidate identifiers:

- `booking_id`: order/booking identifier
- `session_id`: customer session
- `customer_id`: customer identifier
- `product_id`: product identifier

In [40]:
print("Rows:", len(transaction))
print("Unique booking_id:", transaction["booking_id"].nunique())
print("Unique session_id:", transaction["session_id"].nunique())
print("Unique customer_id:", transaction["customer_id"].nunique())
print("Unique product_id:", transaction["product_id"].nunique())

Rows: 1254585
Unique booking_id: 852584
Unique session_id: 852584
Unique customer_id: 50705
Unique product_id: 44446


### 5.3 Booking and Session Relationship

We validate whether:

- One booking can contain multiple sessions.
- One session can contain multiple bookings.

In [ ]:
booking_session = (
    transaction.groupby("booking_id")["session_id"]
    .nunique()
)

print("Bookings:", booking_session.shape[0])
print(
    "Bookings with multiple sessions:",
    (booking_session > 1).sum()
)
print(
    "Maximum sessions per booking:",
    booking_session.max()
)

Bookings: 852584
Bookings with multiple sessions: 0
Maximum sessions per booking: 1


In [ ]:
session_booking = (
    transaction.groupby("session_id")["booking_id"]
    .nunique()
)

print("Sessions:", session_booking.shape[0])
print(
    "Sessions with multiple bookings:",
    (session_booking > 1).sum()
)
print(
    "Maximum bookings per session:",
    session_booking.max()
)

Sessions: 852584
Sessions with multiple bookings: 0
Maximum bookings per session: 1


### Finding

The relationship between `booking_id` and `session_id` is 1:1:

- Each booking belongs to exactly one session.
- Each session belongs to exactly one booking.

### 5.4 Rows per Booking

To determine whether the transaction table is order-level or order-line-level,
we examine the number of rows associated with each booking.

In [ ]:
booking_rows = (
    transaction.groupby("booking_id")
    .size()
    .value_counts()
    .sort_index()
)

print(booking_rows.head(20))

1     706668
2      68100
3      28180
4      16038
5       9791
6       6550
7       4407
8       3393
9       2211
10      1701
11      1363
12       907
13       759
14       609
15       436
16       339
17       260
18       179
19       141
20       120
Name: count, dtype: int64


In [ ]:
print(
    "Bookings with more than 1 row:",
    (transaction.groupby("booking_id").size() > 1).sum()
)

print(
    "Maximum rows per booking:",
    transaction.groupby("booking_id").size().max()
)

Bookings with more than 1 row: 145916
Maximum rows per booking: 51


### Finding

Multiple transaction rows can belong to the same booking.

Therefore:

> The transaction table has an order-line / transaction-line grain.

A booking may contain multiple product lines, with a maximum of 51 rows observed in the dataset.

### 5.5 Data Quality

In [41]:
transaction.isna().sum().sort_values(ascending=False)

promo_code                773446
created_at                     0
booking_id                     0
customer_id                    0
session_id                     0
payment_method                 0
payment_status                 0
promo_amount                   0
shipment_fee                   0
shipment_date_limit            0
shipment_location_lat          0
shipment_location_long         0
total_amount                   0
product_id                     0
quantity                       0
item_price                     0
dtype: int64

In [42]:
transaction.duplicated().sum()

np.int64(0)

#### Finding
- 773,446 missing values in the `promo_code` column.
- No duplicated values

### 5.6 Numeric Validation

In [43]:
print("Quantity <= 0:", (transaction["quantity"] <= 0).sum())
print("Item price <= 0:", (transaction["item_price"] <= 0).sum())
print("Promo amount < 0:", (transaction["promo_amount"] < 0).sum())
print("Shipment fee < 0:", (transaction["shipment_fee"] < 0).sum())
print("Total amount <= 0:", (transaction["total_amount"] <= 0).sum())

Quantity <= 0: 0
Item price <= 0: 0
Promo amount < 0: 0
Shipment fee < 0: 0
Total amount <= 0: 0


### 5.7 Date Validation

In [44]:
transaction["created_at"] = pd.to_datetime(
    transaction["created_at"],
    errors="coerce"
)

print("Invalid created_at:", transaction["created_at"].isna().sum())
print("Min:", transaction["created_at"].min())
print("Max:", transaction["created_at"].max())

Invalid created_at: 0
Min: 2016-06-30 23:18:44.792905+00:00
Max: 2022-07-31 23:59:45.821469+00:00


### 5.8 Business Logic Validation

Because `total_amount` may be repeated across multiple product lines,
we validate whether it represents a booking-level amount.

We also reconstruct the expected total from:

`quantity × item_price`

In [46]:
transaction["line_value"] = (
    transaction["quantity"] * transaction["item_price"]
)

In [48]:
booking_check = (
    transaction
    .groupby("booking_id")
    .agg(
        rows=("booking_id", "size"),
        customer_nunique=("customer_id", "nunique"),
        session_nunique=("session_id", "nunique"),
        quantity_sum=("quantity", "sum"),
        line_value=("line_value", "sum"),
        promo_amount=("promo_amount", "first"),
        shipment_fee=("shipment_fee", "first"),
        total_amount=("total_amount", "first"),
        total_amount_nunique=("total_amount", "nunique")
    )
)

booking_check["calculated_total"] = (
    booking_check["line_value"]
    - booking_check["promo_amount"]
    + booking_check["shipment_fee"]
)

booking_check["difference"] = (
    booking_check["total_amount"]
    - booking_check["calculated_total"]
)


In [49]:
print(booking_check["difference"].describe())

print(
    "Exact matches:",
    (booking_check["difference"] == 0).sum()
)

print(
    "Total bookings:",
    len(booking_check)
)

count    852584.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: difference, dtype: float64
Exact matches: 852584
Total bookings: 852584


### Finding

The transaction amount logic is fully consistent across all bookings.

The validated formula is:

`Line Value = Quantity × Item Price`

`Total Amount = Σ(Line Value) − Promo Amount + Shipment Fee`

All 852,584 bookings match the reconstructed total exactly.

Therefore:

- `total_amount` is a booking-level measure.
- It is repeated across transaction lines.
- Directly summing `total_amount` from the transaction table would cause double counting.

### 5.9 Payment Status Validation

In [ ]:
payment_check = (
    transaction
    .groupby("booking_id")
    .agg(
        payment_status_nunique=("payment_status", "nunique"),
        payment_status=("payment_status", "first")
    )
)

print(
    "Bookings with multiple payment statuses:",
    (payment_check["payment_status_nunique"] > 1).sum()
)

print("\nPayment status distribution:")
print(payment_check["payment_status"].value_counts())

Bookings with multiple payment statuses: 0

Payment status distribution:
payment_status
Success    815964
Failed      36620
Name: count, dtype: int64


In [ ]:
print(transaction["payment_status"].value_counts())

payment_status
Success    1201035
Failed       53550
Name: count, dtype: int64


### Finding

`payment_status` is consistent within each booking.

The dataset contains:

- Success
- Failed

For sales analysis, successful payments will be used as the basis for realized sales metrics.

Failed transactions will be retained in the raw data and handled through filtering/business logic during analysis.

### 5.10 Promotion Logic Validation

In [ ]:
promo_check = (
    transaction
    .groupby("booking_id")
    .agg(
        promo_code_nunique=("promo_code", "nunique"),
        promo_amount_nunique=("promo_amount", "nunique"),
        promo_amount=("promo_amount", "first"),
        promo_code=("promo_code", "first")
    )
)

print(
    "Bookings with multiple promo amounts:",
    (promo_check["promo_amount_nunique"] > 1).sum()
)

print(
    "Bookings with multiple promo codes:",
    (promo_check["promo_code_nunique"] > 1).sum()
)

Bookings with multiple promo amounts: 0
Bookings with multiple promo codes: 0


In [ ]:
print(
    "Promo amount > 0 but promo code missing:",
    (
        (transaction["promo_amount"] > 0)
        & (transaction["promo_code"].isna())
    ).sum()
)

print(
    "Promo code present but promo amount = 0:",
    (
        transaction["promo_code"].notna()
        & (transaction["promo_amount"] == 0)
    ).sum()
)

Promo amount > 0 but promo code missing: 0
Promo code present but promo amount = 0: 0


### Finding

Promotion data is internally consistent:

- Each booking has at most one promo code.
- Each booking has at most one promo amount.
- Every positive promo amount has a corresponding promo code.
- Every promo code has a positive promo amount.

Missing `promo_code` values therefore represent orders without a promotion rather than missing data requiring imputation.

## Transaction Audit Summary

### Confirmed Structure

| Finding | Result |
|---|---|
| Transaction grain | 1 row = 1 product line |
| Booking → Session | 1:1 |
| Booking → Transaction Lines | 1:N |
| Booking-level `total_amount` | Confirmed |
| `quantity × item_price` | Line value |
| Payment status consistency | Confirmed |
| Promotion consistency | Confirmed |
| Invalid quantity | None detected |
| Invalid item price | None detected |
| Invalid `created_at` | None detected |

### Key Modeling Implication

The transaction dataset should be separated conceptually into:

1. **Order-level data**
2. **Order-item-level data**

This prevents double counting of booking-level metrics such as `total_amount`, `promo_amount`, and `shipment_fee`.

### Cleaning Plan

The following transformations will be considered in the cleaning/modeling phase:

- Separate order-level and order-item-level data.
- Preserve raw transaction records.
- Use successful payments for realized sales metrics.
- Handle `promo_code` missing values as "No Promo" only in the processed layer if required.

## 6. Clickstream Table Profiling

The clickstream dataset contains customer interaction events and will support:

- Engagement analysis
- Customer journey analysis
- Funnel analysis
- Conversion analysis

The expected grain is:

> 1 row = 1 customer event

### 6.1 Schema

In [51]:
print("Shape:", click_stream.shape)

print("\nColumns:")
print(click_stream.columns.tolist())

print("\nData types:")
print(click_stream.dtypes)

Shape: (12833602, 12)

Columns:
['session_id', 'event_name', 'event_time', 'event_id', 'traffic_source', 'product_id', 'quantity', 'item_price', 'payment_status', 'search_keywords', 'promo_code', 'promo_amount']

Data types:
session_id             str
event_name             str
event_time             str
event_id               str
traffic_source         str
product_id         float64
quantity           float64
item_price         float64
payment_status         str
search_keywords        str
promo_code             str
promo_amount       float64
dtype: object


### 6.2 Grain and Primary Key

In [6]:
click_stream.head(5)

,session_id,event_name,event_time,event_id,traffic_source,product_id,quantity,item_price,payment_status,search_keywords,promo_code,promo_amount
0,fb0abf9e-fd1a-44dd-b5c0-2834d5a4b81c,HOMEPAGE,2019-09-06T15:54:32.821085Z,9c4388c4-c95b-4678-b5ca-e9cbc0734109,MOBILE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fb0abf9e-fd1a-44dd-b5c0-2834d5a4b81c,SCROLL,2019-09-06T16:03:57.821085Z,4690e1f5-3f99-42d3-84a5-22c4c4d8500a,MOBILE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7d440441-e67a-4d36-b324-80ffd636d166,HOMEPAGE,2019-09-01T12:05:10.322763Z,88aeaeb5-ec98-4859-852c-8abb483faf31,MOBILE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7d440441-e67a-4d36-b324-80ffd636d166,ADD_TO_CART,2019-09-01T12:06:33.322763Z,934e306e-ecc6-472f-9ccb-12c8536910a2,MOBILE,15315.0,4.0,313529.0,NaN,NaN,NaN,NaN
4,7d440441-e67a-4d36-b324-80ffd636d166,BOOKING,2019-09-01T12:15:29.425431Z,9f4767a1-40fa-4c9c-9524-dfad18634d56,MOBILE,NaN,NaN,NaN,Success,NaN,NaN,NaN


In [7]:
print("Rows:", len(click_stream))
print("Unique session_id:", click_stream["session_id"].nunique())
print("Unique event_id:", click_stream["event_id"].nunique())

Rows: 12833602
Unique session_id: 895203
Unique event_id: 12833602


### Initial Finding

The clickstream dataset contains 12,833,602 event records across 895,203 sessions.

- `event_id` is unique across all rows and is therefore a candidate primary key.
- `session_id` is not unique because each session contains multiple events.

### Initial Grain

> 1 row = 1 customer interaction event

### 6.3 Session and Event Relationship


In [54]:
session_events = (
    click_stream.groupby("session_id")
    .size()
    .value_counts()
    .sort_index()
)

print(session_events.head(20))

2      4794
3      8218
4     90348
5     83283
6     68265
7     59056
8     52737
9     47569
10    43231
11    38980
12    34935
13    32251
14    29115
15    26203
16    23959
17    21661
18    19636
19    17951
20    16312
21    14695
Name: count, dtype: int64


In [9]:
session_event_count = click_stream.groupby("session_id").size()

print("Sessions with more than 1 event:",
      (session_event_count > 1).sum())

print("Maximum events per session:",
      session_event_count.max())

print("Average events per session:",
      session_event_count.mean())

Sessions with more than 1 event: 895203
Maximum events per session: 650
Average events per session: 14.335968489828565


### Session-Level Event Distribution

The clickstream data contains 895,203 unique sessions.

Key findings:

- All 895,203 sessions contain more than one event.
- The average session contains approximately 14.34 events.
- The maximum number of events within a single session is 650.
- `session_id` is therefore a session-level identifier rather than an event-level key.

### Confirmed Grain

> 1 row = 1 customer interaction event

Relationship:

`session_id` 1:N `event_id`

This structure allows the clickstream data to support session-level engagement,
customer journey, and funnel analysis.

### 6.4 Event Distribution

In [10]:
click_stream["event_name"].value_counts()

event_name
CLICK          2498038
HOMEPAGE       2487126
ADD_TO_CART    1937157
SCROLL         1663571
ITEM_DETAIL    1290396
SEARCH         1173266
BOOKING         852582
PROMO_PAGE      604931
ADD_PROMO       326535
Name: count, dtype: int64

### Event Type Distribution

The clickstream dataset contains 9 distinct event types representing different
customer interactions throughout the e-commerce journey.

The most frequent events are:

- `CLICK`
- `HOMEPAGE`
- `ADD_TO_CART`
- `SCROLL`
- `ITEM_DETAIL`
- `SEARCH`

The dataset also contains conversion-related events such as:

- `BOOKING`
- `ADD_PROMO`

### Initial Finding

The clickstream data captures multiple stages of the customer journey, from
general browsing and product interaction to cart activity, promotion usage,
and booking.

This makes the dataset suitable for:

- Engagement analysis
- Customer journey analysis
- Funnel analysis
- Conversion analysis

### Data Quality Note

The number of `BOOKING` events (852,582) is very close to the number of unique
bookings identified in the transaction dataset (852,584).

This relationship will be validated further during the data integration and
modeling phase.

### 6.5 Data Quality

In [55]:
click_stream.isna().sum().sort_index(ascending=False)

traffic_source            0
session_id                0
search_keywords    11660336
quantity           10896445
promo_code         12507067
promo_amount       12507067
product_id         10896445
payment_status     11981020
item_price         10896445
event_time                0
event_name                0
event_id                  0
dtype: int64

In [56]:
click_stream.duplicated().sum()

np.int64(0)

### 6.6 Event-Specific Missing Values

In [57]:
important_columns = [
    "product_id",
    "quantity",
    "item_price",
    "payment_status",
    "search_keywords",
    "promo_code",
    "promo_amount"
]

event_non_null = (
    click_stream
    .groupby("event_name")[important_columns]
    .count()
)

event_non_null

,product_id,quantity,item_price,payment_status,search_keywords,promo_code,promo_amount
event_name,,,,,,,
ADD_PROMO,0,0,0,0,0,326535,326535
ADD_TO_CART,1937157,1937157,1937157,0,0,0,0
BOOKING,0,0,0,852582,0,0,0
CLICK,0,0,0,0,0,0,0
HOMEPAGE,0,0,0,0,0,0,0
ITEM_DETAIL,0,0,0,0,0,0,0
PROMO_PAGE,0,0,0,0,0,0,0
SCROLL,0,0,0,0,0,0,0
SEARCH,0,0,0,0,1173266,0,0


### Event-Level Missing Value Analysis

Missing values in the clickstream data are not randomly distributed across
the dataset. Instead, their presence depends largely on the event type.

The core event-tracking fields (`session_id`, `event_time`, `event_id`, and
`traffic_source`) are complete across the dataset.

Other attributes are populated selectively depending on the type of customer
interaction. For example:

- `SEARCH` events contain `search_keywords`.
- `ADD_TO_CART` events contain product and transaction-related attributes.
- `BOOKING` events contain booking/payment-related attributes.
- Browsing events such as `HOMEPAGE`, `CLICK`, and `SCROLL` do not necessarily
  require product or promotion attributes.

### Finding

The missing values are primarily structural rather than random missing data.

Therefore, these values should not be blindly imputed during the cleaning
phase.

### Treatment

Event-specific missing values will be preserved unless they violate the
expected business logic of that event type.

The cleaning process will focus on detecting unexpected values within each
event type rather than filling all missing values globally.

### 6.7 Booking Event Validation

In [11]:
booking_events = click_stream[
    click_stream["event_name"] == "BOOKING"
]

print("BOOKING events:", len(booking_events))
print("Unique sessions:", booking_events["session_id"].nunique())

BOOKING events: 852582
Unique sessions: 852582


In [12]:
booking_sessions = (
    booking_events
    .groupby("session_id")
    .size()
)

print("Sessions with multiple BOOKING events:",
      (booking_sessions > 1).sum())

print("Maximum BOOKING events per session:",
      booking_sessions.max())

Sessions with multiple BOOKING events: 0
Maximum BOOKING events per session: 1


### Booking Event Validation

The `BOOKING` event was further validated at the session level.

Findings:

- 852,582 `BOOKING` events were recorded.
- These events belong to 852,582 unique sessions.
- No session contains more than one `BOOKING` event.
- Therefore, each session contains at most one booking event.

Combined with the transaction audit, where `booking_id` and `session_id`
have a 1:1 relationship, the clickstream and transaction datasets can be
linked through `session_id` for customer journey and conversion analysis.

> Note: The number of `BOOKING` events (852,582) differs slightly from the
> number of unique transaction bookings (852,584). This difference will be
> investigated during data integration.

### 6.8 Event Time Validation

In [58]:
click_stream["event_time"] = pd.to_datetime(
    click_stream["event_time"],
    errors="coerce"
)

print(
    "Invalid event_time:",
    click_stream["event_time"].isna().sum()
)

print(
    "Min event_time:",
    click_stream["event_time"].min()
)

print(
    "Max event_time:",
    click_stream["event_time"].max()
)

Invalid event_time: 0
Min event_time: 2016-06-30 22:59:36.254358+00:00
Max event_time: 2022-08-01 00:49:59.408424+00:00


# 7. Profiling Summary

## Confirmed Data Structure

| Dataset | Grain | Candidate Key |
|---|---|---|
| Customer | 1 row = 1 customer | `customer_id` |
| Product | 1 row = 1 product | `id` |
| Transaction | 1 row = 1 product line | `booking_id` + product line |
| Clickstream | 1 row = 1 event | `event_id` |

## Confirmed Relationships

- Customer → Transaction: 1:N
- Customer ↔ Device: 1:1
- Product → Transaction: 1:N
- Booking → Transaction Lines: 1:N
- Booking ↔ Session: 1:1
- Session → Clickstream Events: 1:N

## Key Data Quality Findings

1. `product.csv` contains 22 malformed rows caused by unescaped commas
   in `productDisplayName`.
2. Transaction-level booking amounts are internally consistent.
3. `payment_status` is consistent at booking level.
4. Clickstream missing values are largely event-specific.
5. `BOOKING` events are unique at session level.
6. Clickstream and transaction booking counts differ by 2 records.

## Next Step

The profiling results will be used to define the data cleaning and
transformation requirements for the next stage.